# SEAD 670 - Sustainable AI Deployment Challenge

## Student role
You are sustainability analysts and deployment designers. Your client wants to deploy an already-built AI text-classification service. Your job is to decide how the service should be configured and operated to reduce energy, carbon, and cost while maintaining acceptable performance.

You will not train models, write PyTorch, implement metrics, install packages, or design neural architectures. The AI system is supplied as a ready-to-use engineering system. You will change configuration choices, run controlled experiments, compare outcomes, and defend a deployment recommendation.


## How to use this notebook

1. Run **System setup** once.
2. Run **Launch AI Deployment Workbench**.
3. Use the tabs to run experiments, save results, compare configurations, simulate deployments, identify Pareto-efficient options, and apply client scenarios.
4. If the interactive buttons do not display in your browser, use the fallback Colab form cells near the bottom of the notebook.

The numbers in this notebook are deterministic teaching estimates for a supplied text-classification service. They are designed to support engineering reasoning about tradeoffs. For a real procurement or deployment decision, replace the embedded carbon intensity, price, and hardware assumptions with current measured values from the target organization.


In [ ]:
#@title System setup - run once
import hashlib
import itertools
import math
import os
from datetime import datetime

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, clear_output, display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
    WIDGETS_ERROR = None
except Exception as exc:
    WIDGETS_AVAILABLE = False
    WIDGETS_ERROR = repr(exc)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

MODEL_OPTIONS = ['Small', 'Medium', 'Large']
QUANT_OPTIONS = ['FP16', 'INT8', 'INT4']
POOL_OPTIONS = ['Last-token', 'Mean', 'Attention']
BATCH_OPTIONS = [1, 8, 32]
INPUT_OPTIONS = ['Clean', 'Mild noise', 'Heavy noise']

MODEL_PROFILES = {
    'Small': {
        'base_f1': 0.922,
        'latency_ms': 14.0,
        'energy_wh_per_1k': 0.48,
        'memory_gb': 1.3,
        'hardware_usd_per_hour': 0.12,
        'hardware_tier': 'CPU or small accelerator',
        'description': 'Lowest memory and compute demand; may lose accuracy on difficult inputs.',
    },
    'Medium': {
        'base_f1': 0.953,
        'latency_ms': 28.0,
        'energy_wh_per_1k': 0.92,
        'memory_gb': 3.0,
        'hardware_usd_per_hour': 0.45,
        'hardware_tier': 'single mid-range accelerator',
        'description': 'Balanced quality, latency, energy, and cost.',
    },
    'Large': {
        'base_f1': 0.972,
        'latency_ms': 55.0,
        'energy_wh_per_1k': 1.75,
        'memory_gb': 7.0,
        'hardware_usd_per_hour': 1.25,
        'hardware_tier': 'larger GPU-class accelerator',
        'description': 'Highest quality, but more memory, energy, and compute cost.',
    },
}

QUANT_PROFILES = {
    'FP16': {'f1_delta': 0.000, 'latency_factor': 1.00, 'energy_factor': 1.00, 'memory_factor': 1.00, 'rate_factor': 1.00},
    'INT8': {'f1_delta': -0.004, 'latency_factor': 0.72, 'energy_factor': 0.62, 'memory_factor': 0.55, 'rate_factor': 0.86},
    'INT4': {'f1_delta': -0.018, 'latency_factor': 0.58, 'energy_factor': 0.42, 'memory_factor': 0.32, 'rate_factor': 0.74},
}

POOL_PROFILES = {
    'Last-token': {'f1_delta': -0.004, 'latency_factor': 0.92, 'energy_factor': 0.93, 'memory_extra_gb': 0.00},
    'Mean': {'f1_delta': 0.000, 'latency_factor': 1.00, 'energy_factor': 1.00, 'memory_extra_gb': 0.00},
    'Attention': {'f1_delta': 0.006, 'latency_factor': 1.18, 'energy_factor': 1.13, 'memory_extra_gb': 0.15},
}

INPUT_PROFILES = {
    'Clean': {'f1_delta': 0.000, 'latency_factor': 1.00, 'energy_factor': 1.00},
    'Mild noise': {'f1_delta': -0.012, 'latency_factor': 1.03, 'energy_factor': 1.04},
    'Heavy noise': {'f1_delta': -0.040, 'latency_factor': 1.08, 'energy_factor': 1.10},
}

BATCH_PROFILES = {
    1: {'latency_factor': 1.00, 'energy_factor': 1.00, 'memory_extra_gb': 0.10},
    8: {'latency_factor': 1.25, 'energy_factor': 0.70, 'memory_extra_gb': 0.35},
    32: {'latency_factor': 2.40, 'energy_factor': 0.55, 'memory_extra_gb': 0.95},
}

GRID_PROFILES = {
    'Quebec': {'kg_co2e_per_kwh': 0.0017, 'electricity_usd_per_kwh': 0.073, 'note': 'Very low-carbon hydro-dominated teaching profile.'},
    'Ontario': {'kg_co2e_per_kwh': 0.0300, 'electricity_usd_per_kwh': 0.090, 'note': 'Low-carbon mixed-grid teaching profile.'},
    'Alberta': {'kg_co2e_per_kwh': 0.5400, 'electricity_usd_per_kwh': 0.115, 'note': 'Carbon-intensive teaching profile.'},
    'Custom': {'kg_co2e_per_kwh': 0.2000, 'electricity_usd_per_kwh': 0.100, 'note': 'Use your own grid carbon intensity and electricity price.'},
}

SCENARIOS = {
    'Custom': {
        'requests_per_day': 1_000_000,
        'grid': 'Ontario',
        'input_condition': 'Clean',
        'max_latency_ms': 100,
        'priority': 'Balanced',
        'required_f1_percent': 93.0,
        'rule': 'fixed',
        'note': 'Set your own assumptions and constraints.',
    },
    'Hospital document-routing system': {
        'requests_per_day': 50_000,
        'grid': 'Ontario',
        'input_condition': 'Heavy noise',
        'max_latency_ms': 120,
        'priority': 'Maximize performance',
        'required_f1_percent': None,
        'rule': 'at least 98% of best F1',
        'note': 'Moderate volume; accuracy is the binding constraint.',
    },
    'High-volume e-commerce classifier': {
        'requests_per_day': round(100_000_000 / 365),
        'grid': 'Ontario',
        'input_condition': 'Clean',
        'max_latency_ms': 100,
        'priority': 'Minimize cost',
        'required_f1_percent': None,
        'rule': 'within 1.0 F1 point of best',
        'note': 'Very high traffic; small efficiency gains compound.',
    },
    'Small local manufacturer': {
        'requests_per_day': 300,
        'grid': 'Ontario',
        'input_condition': 'Mild noise',
        'max_latency_ms': 250,
        'priority': 'Low hardware',
        'required_f1_percent': 90.0,
        'rule': 'fixed',
        'note': 'Low volume and local deployment make memory and hardware simplicity important.',
    },
    'Public-sector service in Quebec': {
        'requests_per_day': 80_000,
        'grid': 'Quebec',
        'input_condition': 'Clean',
        'max_latency_ms': 100,
        'priority': 'Minimize carbon',
        'required_f1_percent': 93.0,
        'rule': 'fixed',
        'note': 'Low-carbon electricity, but public accountability favors efficient choices.',
    },
    'Cloud application on carbon-intensive grid': {
        'requests_per_day': 2_000_000,
        'grid': 'Alberta',
        'input_condition': 'Mild noise',
        'max_latency_ms': 100,
        'priority': 'Minimize carbon',
        'required_f1_percent': 93.0,
        'rule': 'fixed',
        'note': 'Large traffic on a carbon-intensive grid makes energy efficiency environmentally significant.',
    },
}

EXPERIMENT_TEMPLATES = {
    'Group A - Model size': {
        'question': 'When is additional model complexity worth paying for?',
        'configs': [{'Model': m, 'Quantization': 'INT8', 'Pooling': 'Mean', 'Batch size': 8, 'Input condition': 'Clean'} for m in MODEL_OPTIONS],
    },
    'Group B - Quantization': {
        'question': 'How much computational cost can be removed before quality degrades too much?',
        'configs': [{'Model': 'Medium', 'Quantization': q, 'Pooling': 'Mean', 'Batch size': 8, 'Input condition': 'Clean'} for q in QUANT_OPTIONS],
    },
    'Group C - Batching': {
        'question': 'How does serving many requests together affect energy, latency, and throughput?',
        'configs': [{'Model': 'Medium', 'Quantization': 'INT8', 'Pooling': 'Mean', 'Batch size': b, 'Input condition': 'Clean'} for b in BATCH_OPTIONS],
    },
    'Group D - Robustness': {
        'question': 'Does the most energy-efficient model remain effective when real inputs become messy?',
        'configs': [{'Model': 'Medium', 'Quantization': 'INT8', 'Pooling': p, 'Batch size': 8, 'Input condition': i} for p in POOL_OPTIONS for i in INPUT_OPTIONS],
    },
    'Group E - Deployment scale': {
        'question': 'When does a small efficiency improvement become environmentally important?',
        'configs': [
            {'Model': 'Small', 'Quantization': 'INT8', 'Pooling': 'Mean', 'Batch size': 8, 'Input condition': 'Clean'},
            {'Model': 'Medium', 'Quantization': 'INT8', 'Pooling': 'Mean', 'Batch size': 8, 'Input condition': 'Clean'},
            {'Model': 'Medium', 'Quantization': 'INT4', 'Pooling': 'Mean', 'Batch size': 32, 'Input condition': 'Clean'},
            {'Model': 'Large', 'Quantization': 'FP16', 'Pooling': 'Attention', 'Batch size': 8, 'Input condition': 'Clean'},
        ],
    },
}

GLOSSARY = {
    'Model size': 'Small, Medium, and Large represent increasingly complex supplied classifiers. Larger models usually improve F1 but require more memory and compute.',
    'Quantization': 'A lower-bit representation of model weights and activations. INT8 and INT4 usually reduce memory and computation, sometimes with lower F1.',
    'FP16': '16-bit floating point representation. High quality and common on GPUs, but more memory and energy than lower-bit options.',
    'INT8': '8-bit integer representation. Usually a strong efficiency option with modest quality loss.',
    'INT4': '4-bit integer representation. Very compact and efficient, but quality can degrade on difficult inputs.',
    'Pooling': 'How the text representation is summarized before classification. Last-token is cheap, mean is balanced, attention is more adaptive but more expensive.',
    'Batch size': 'How many requests are processed together. Larger batches often improve throughput and energy per request but can increase user-facing latency.',
    'F1': 'A classification quality metric that balances precision and recall. Higher is better.',
    'Latency': 'Time for a request or batch to return. Lower is better for interactive services.',
    'Throughput': 'How many requests the service can process per second. Higher is better for high-volume systems.',
    'Energy per 1,000 requests': 'Estimated electricity used to serve 1,000 classification requests.',
    'Pareto-efficient': 'A configuration is Pareto-efficient when no other configuration is both at least as accurate and at least as energy-efficient, with one strictly better.',
}

SAVED_RESULTS = []
LATEST_RESULT = None


def stable_adjustment(*items, scale=0.0015):
    text = '|'.join(str(item) for item in items)
    digest = hashlib.sha256(text.encode('utf-8')).hexdigest()
    value = int(digest[:8], 16) / 0xFFFFFFFF
    return (value - 0.5) * 2 * scale


def evaluate_config(model, quantization, pooling, batch_size, input_condition):
    model_p = MODEL_PROFILES[model]
    quant_p = QUANT_PROFILES[quantization]
    pool_p = POOL_PROFILES[pooling]
    input_p = INPUT_PROFILES[input_condition]
    batch_p = BATCH_PROFILES[int(batch_size)]

    f1 = model_p['base_f1'] + quant_p['f1_delta'] + pool_p['f1_delta'] + input_p['f1_delta']

    if input_condition == 'Mild noise' and pooling == 'Attention':
        f1 += 0.005
    if input_condition == 'Heavy noise' and pooling == 'Attention':
        f1 += 0.014
    if input_condition == 'Mild noise' and pooling == 'Last-token':
        f1 -= 0.004
    if input_condition == 'Heavy noise' and pooling == 'Last-token':
        f1 -= 0.015
    if input_condition == 'Heavy noise' and pooling == 'Mean':
        f1 -= 0.004
    if input_condition == 'Heavy noise' and quantization == 'INT8':
        f1 -= 0.003
    if input_condition == 'Heavy noise' and quantization == 'INT4':
        f1 -= 0.012
    if model == 'Small' and quantization == 'INT4':
        f1 -= 0.006
    if model == 'Large' and quantization == 'FP16' and pooling == 'Attention':
        f1 += 0.002

    f1 += stable_adjustment(model, quantization, pooling, batch_size, input_condition)
    f1 = min(max(f1, 0.78), 0.989)
    accuracy = min(max(f1 + 0.003 + stable_adjustment('accuracy', model, quantization, pooling, batch_size, input_condition, scale=0.001), 0.78), 0.992)

    latency_ms = model_p['latency_ms'] * quant_p['latency_factor'] * pool_p['latency_factor'] * input_p['latency_factor'] * batch_p['latency_factor']
    latency_ms = latency_ms * (1 + stable_adjustment('latency', model, quantization, pooling, batch_size, input_condition, scale=0.025))

    throughput = int((int(batch_size) * 1000.0 / latency_ms) * 0.92)

    energy_wh_per_1k = model_p['energy_wh_per_1k'] * quant_p['energy_factor'] * pool_p['energy_factor'] * input_p['energy_factor'] * batch_p['energy_factor']
    energy_wh_per_1k = energy_wh_per_1k * (1 + stable_adjustment('energy', model, quantization, pooling, batch_size, input_condition, scale=0.03))

    memory_gb = model_p['memory_gb'] * quant_p['memory_factor'] + pool_p['memory_extra_gb'] + batch_p['memory_extra_gb']
    memory_gb = memory_gb * (1 + stable_adjustment('memory', model, quantization, pooling, batch_size, input_condition, scale=0.015))

    hourly_rate = model_p['hardware_usd_per_hour'] * quant_p['rate_factor'] * (1.05 if pooling == 'Attention' else 1.0)

    return {
        'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Model': model,
        'Quantization': quantization,
        'Pooling': pooling,
        'Batch size': int(batch_size),
        'Input condition': input_condition,
        'Accuracy_percent': round(accuracy * 100, 2),
        'F1_percent': round(f1 * 100, 2),
        'Latency_ms': round(latency_ms, 1),
        'Throughput_requests_per_s': throughput,
        'Energy_Wh_per_1000_requests': round(energy_wh_per_1k, 3),
        'Memory_GB': round(memory_gb, 2),
        'Hardware tier': model_p['hardware_tier'],
        'Estimated compute USD_per_hour': round(hourly_rate, 3),
    }


def all_configurations(input_condition=None):
    rows = []
    input_values = [input_condition] if input_condition else INPUT_OPTIONS
    for model, quantization, pooling, batch_size, condition in itertools.product(MODEL_OPTIONS, QUANT_OPTIONS, POOL_OPTIONS, BATCH_OPTIONS, input_values):
        rows.append(evaluate_config(model, quantization, pooling, batch_size, condition))
    return pd.DataFrame(rows)


def project_deployment(result, requests_per_day=1_000_000, grid='Ontario', years=1.0, custom_kg_co2e_per_kwh=None, custom_electricity_usd_per_kwh=None):
    grid_profile = dict(GRID_PROFILES[grid])
    if grid == 'Custom':
        if custom_kg_co2e_per_kwh is not None:
            grid_profile['kg_co2e_per_kwh'] = float(custom_kg_co2e_per_kwh)
        if custom_electricity_usd_per_kwh is not None:
            grid_profile['electricity_usd_per_kwh'] = float(custom_electricity_usd_per_kwh)

    total_requests = float(requests_per_day) * 365.0 * float(years)
    energy_kwh = result['Energy_Wh_per_1000_requests'] * (total_requests / 1000.0) / 1000.0
    co2e_kg = energy_kwh * grid_profile['kg_co2e_per_kwh']
    compute_hours = total_requests / max(result['Throughput_requests_per_s'], 1) / 3600.0
    compute_cost = compute_hours * result['Estimated compute USD_per_hour']
    electricity_cost = energy_kwh * grid_profile['electricity_usd_per_kwh']

    return {
        'Traffic requests/day': int(requests_per_day),
        'Operating period years': float(years),
        'Grid': grid,
        'Grid kg CO2e/kWh': grid_profile['kg_co2e_per_kwh'],
        'Period energy kWh': round(energy_kwh, 2),
        'Period CO2e kg': round(co2e_kg, 3),
        'Estimated compute cost USD': round(compute_cost, 2),
        'Estimated electricity cost USD': round(electricity_cost, 2),
        'Grid note': grid_profile['note'],
    }


def metric_table(result, requests_per_day=1_000_000, grid='Ontario', years=1.0, custom_kg_co2e_per_kwh=None, custom_electricity_usd_per_kwh=None):
    projection = project_deployment(result, requests_per_day, grid, years, custom_kg_co2e_per_kwh, custom_electricity_usd_per_kwh)
    rows = [
        ('Accuracy', f"{result['Accuracy_percent']:.2f}%"),
        ('F1', f"{result['F1_percent']:.2f}%"),
        ('Latency', f"{result['Latency_ms']:.1f} ms"),
        ('Throughput', f"{result['Throughput_requests_per_s']:,} requests/s"),
        ('Energy', f"{result['Energy_Wh_per_1000_requests']:.3f} Wh / 1,000 requests"),
        ('Memory', f"{result['Memory_GB']:.2f} GB"),
        ('Estimated period energy', f"{projection['Period energy kWh']:.2f} kWh"),
        ('Estimated period CO2e', f"{projection['Period CO2e kg']:.3f} kg"),
        ('Estimated compute cost', f"${projection['Estimated compute cost USD']:,.2f}"),
        ('Hardware tier', result['Hardware tier']),
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Result'])


def compact_results_table(rows):
    if not rows:
        return pd.DataFrame()
    table = pd.DataFrame(rows)
    cols = []
    if 'Run ID' in table.columns:
        cols.append('Run ID')
    cols.extend([
        'Model', 'Quantization', 'Pooling', 'Batch size', 'Input condition',
        'F1_percent', 'Latency_ms', 'Throughput_requests_per_s',
        'Energy_Wh_per_1000_requests', 'Memory_GB', 'Hardware tier'
    ])
    return table[cols]


def save_result(result):
    global SAVED_RESULTS
    if result is None:
        return False
    saved = dict(result)
    saved['Run ID'] = len(SAVED_RESULTS) + 1
    SAVED_RESULTS.append(saved)
    return True


def pareto_flags(df, quality_col='F1_percent', impact_col='Energy_Wh_per_1000_requests'):
    flags = []
    for _, row in df.iterrows():
        dominated = ((df[quality_col] >= row[quality_col]) & (df[impact_col] <= row[impact_col]) & ((df[quality_col] > row[quality_col]) | (df[impact_col] < row[impact_col]))).any()
        flags.append(not dominated)
    return flags


def scenario_required_f1_percent(df, scenario_name, fallback_percent):
    scenario = SCENARIOS[scenario_name]
    best = df['F1_percent'].max()
    if scenario['rule'] == 'at least 98% of best F1':
        return round(best * 0.98, 2)
    if scenario['rule'] == 'within 1.0 F1 point of best':
        return round(best - 1.0, 2)
    return float(fallback_percent)


def rank_configurations(df, requests_per_day, years, grid, required_f1_percent, max_latency_ms, priority, custom_kg_co2e_per_kwh=None, custom_electricity_usd_per_kwh=None):
    projected_rows = []
    for _, row in df.iterrows():
        result = row.to_dict()
        projection = project_deployment(result, requests_per_day, grid, years, custom_kg_co2e_per_kwh, custom_electricity_usd_per_kwh)
        combined = {**result, **projection}
        projected_rows.append(combined)

    out = pd.DataFrame(projected_rows)
    out['Meets F1'] = out['F1_percent'] >= required_f1_percent
    out['Meets latency'] = out['Latency_ms'] <= max_latency_ms
    out['Feasible'] = out['Meets F1'] & out['Meets latency']

    def normalize_good(series):
        if series.max() == series.min():
            return pd.Series([1.0] * len(series), index=series.index)
        return (series - series.min()) / (series.max() - series.min())

    def normalize_low(series):
        if series.max() == series.min():
            return pd.Series([1.0] * len(series), index=series.index)
        return 1 - (series - series.min()) / (series.max() - series.min())

    f1_score = normalize_good(out['F1_percent'])
    energy_score = normalize_low(out['Period energy kWh'])
    carbon_score = normalize_low(out['Period CO2e kg'])
    latency_score = normalize_low(out['Latency_ms'])
    cost_score = normalize_low(out['Estimated compute cost USD'])
    memory_score = normalize_low(out['Memory_GB'])

    weights = {
        'Balanced': {'f1': 0.35, 'energy': 0.20, 'carbon': 0.10, 'latency': 0.15, 'cost': 0.10, 'memory': 0.10},
        'Minimize carbon': {'f1': 0.20, 'energy': 0.15, 'carbon': 0.45, 'latency': 0.05, 'cost': 0.10, 'memory': 0.05},
        'Minimize cost': {'f1': 0.20, 'energy': 0.15, 'carbon': 0.05, 'latency': 0.10, 'cost': 0.40, 'memory': 0.10},
        'Maximize performance': {'f1': 0.60, 'energy': 0.10, 'carbon': 0.05, 'latency': 0.15, 'cost': 0.05, 'memory': 0.05},
        'Low hardware': {'f1': 0.20, 'energy': 0.20, 'carbon': 0.05, 'latency': 0.05, 'cost': 0.15, 'memory': 0.35},
    }[priority]

    out['Advisor score'] = (
        weights['f1'] * f1_score +
        weights['energy'] * energy_score +
        weights['carbon'] * carbon_score +
        weights['latency'] * latency_score +
        weights['cost'] * cost_score +
        weights['memory'] * memory_score
    )
    out.loc[~out['Feasible'], 'Advisor score'] -= 10
    out = out.sort_values(['Feasible', 'Advisor score', 'F1_percent'], ascending=[False, False, False]).reset_index(drop=True)
    out['Rank'] = np.arange(1, len(out) + 1)
    return out


def display_glossary():
    html = '<details><summary><b>Glossary and tooltips</b></summary><table>'
    for term, explanation in GLOSSARY.items():
        html += f'<tr><td style="padding:4px 12px 4px 0;"><b>{term}</b></td><td style="padding:4px;">{explanation}</td></tr>'
    html += '</table></details>'
    display(HTML(html))


def show_template(template_name):
    template = EXPERIMENT_TEMPLATES[template_name]
    display(Markdown(f"**Question:** {template['question']}"))
    display(pd.DataFrame(template['configs']))


def load_template_results(template_name):
    global SAVED_RESULTS, LATEST_RESULT
    rows = []
    for config in EXPERIMENT_TEMPLATES[template_name]['configs']:
        result = evaluate_config(config['Model'], config['Quantization'], config['Pooling'], config['Batch size'], config['Input condition'])
        result['Run ID'] = len(SAVED_RESULTS) + len(rows) + 1
        rows.append(result)
    SAVED_RESULTS.extend(rows)
    LATEST_RESULT = rows[-1] if rows else LATEST_RESULT
    return rows


def plot_saved_results(df):
    if df.empty:
        display(Markdown('No saved results yet. Run and save experiments first.'))
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(df['Energy_Wh_per_1000_requests'], df['F1_percent'], s=80)
    for idx, row in df.iterrows():
        axes[0].annotate(str(row.get('Run ID', idx + 1)), (row['Energy_Wh_per_1000_requests'], row['F1_percent']), xytext=(5, 5), textcoords='offset points')
    axes[0].set_xlabel('Energy (Wh / 1,000 requests)')
    axes[0].set_ylabel('F1 (%)')
    axes[0].set_title('Accuracy-energy tradeoff')
    axes[0].grid(True, alpha=0.3)

    x = np.arange(len(df))
    axes[1].bar(x - 0.2, df['Latency_ms'], width=0.4, label='Latency ms')
    axes[1].bar(x + 0.2, df['Energy_Wh_per_1000_requests'] * 100, width=0.4, label='Energy x100')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([str(v) for v in df.get('Run ID', range(1, len(df) + 1))], rotation=0)
    axes[1].set_title('Operational metrics by saved run')
    axes[1].set_xlabel('Run ID')
    axes[1].legend()
    axes[1].grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


def launch_workbench():
    if not WIDGETS_AVAILABLE:
        display(Markdown(f"ipywidgets is not available in this runtime: `{WIDGETS_ERROR}`. Use the fallback Colab form cells below."))
        return

    display(Markdown('## AI Deployment Workbench'))
    display_glossary()

    style = {'description_width': '145px'}
    layout = widgets.Layout(width='360px')

    model_w = widgets.Dropdown(options=MODEL_OPTIONS, value='Medium', description='Model', style=style, layout=layout)
    quant_w = widgets.Dropdown(options=QUANT_OPTIONS, value='INT8', description='Quantization', style=style, layout=layout)
    pool_w = widgets.Dropdown(options=POOL_OPTIONS, value='Mean', description='Pooling', style=style, layout=layout)
    batch_w = widgets.Dropdown(options=BATCH_OPTIONS, value=8, description='Batch size', style=style, layout=layout)
    input_w = widgets.Dropdown(options=INPUT_OPTIONS, value='Clean', description='Input condition', style=style, layout=layout)
    run_button = widgets.Button(description='Run Experiment', button_style='primary', icon='play')
    save_button = widgets.Button(description='Save Result', icon='save')
    exp_out = widgets.Output()

    def on_run(_):
        global LATEST_RESULT
        with exp_out:
            clear_output(wait=True)
            LATEST_RESULT = evaluate_config(model_w.value, quant_w.value, pool_w.value, batch_w.value, input_w.value)
            display(Markdown('### Experiment result'))
            display(metric_table(LATEST_RESULT))
            display(Markdown(f"**Interpretation prompt:** What changed because this configuration uses **{model_w.value}**, **{quant_w.value}**, **{pool_w.value} pooling**, and batch size **{batch_w.value}**?"))

    def on_save(_):
        with exp_out:
            if save_result(LATEST_RESULT):
                display(Markdown(f"Saved result {len(SAVED_RESULTS)}."))
            else:
                display(Markdown('Run an experiment before saving.'))

    run_button.on_click(on_run)
    save_button.on_click(on_save)
    experiment_tab = widgets.VBox([
        widgets.HTML('<h3>Choose configuration</h3>'),
        model_w, quant_w, pool_w, batch_w, input_w,
        widgets.HBox([run_button, save_button]),
        exp_out,
    ])

    compare_button = widgets.Button(description='Compare Experiments', button_style='primary', icon='table')
    clear_button = widgets.Button(description='Clear Saved Results', icon='trash')
    compare_out = widgets.Output()

    def on_compare(_):
        with compare_out:
            clear_output(wait=True)
            df = compact_results_table(SAVED_RESULTS)
            if df.empty:
                display(Markdown('No saved results yet.'))
                return
            display(df)
            plot_saved_results(df)

    def on_clear(_):
        global SAVED_RESULTS
        SAVED_RESULTS = []
        with compare_out:
            clear_output(wait=True)
            display(Markdown('Saved results cleared.'))

    compare_button.on_click(on_compare)
    clear_button.on_click(on_clear)
    results_tab = widgets.VBox([
        widgets.HTML('<h3>Saved results</h3>'),
        widgets.HBox([compare_button, clear_button]),
        compare_out,
    ])

    scenario_w = widgets.Dropdown(options=list(SCENARIOS.keys()), value='Custom', description='Client scenario', style=style, layout=widgets.Layout(width='520px'))
    traffic_w = widgets.IntText(value=1_000_000, description='Requests/day', style=style, layout=layout)
    grid_w = widgets.Dropdown(options=list(GRID_PROFILES.keys()), value='Ontario', description='Grid', style=style, layout=layout)
    custom_co2_w = widgets.FloatText(value=0.200, description='Custom kg CO2e/kWh', style=style, layout=layout)
    custom_price_w = widgets.FloatText(value=0.100, description='Custom $/kWh', style=style, layout=layout)
    years_w = widgets.FloatSlider(value=1.0, min=0.25, max=5.0, step=0.25, description='Years', readout_format='.2f', style=style, layout=layout)
    required_f1_w = widgets.FloatSlider(value=93.0, min=80.0, max=99.0, step=0.1, description='Required F1 %', readout_format='.1f', style=style, layout=layout)
    max_latency_w = widgets.IntText(value=100, description='Max latency ms', style=style, layout=layout)
    deploy_input_w = widgets.Dropdown(options=INPUT_OPTIONS, value='Clean', description='Input condition', style=style, layout=layout)
    priority_w = widgets.Dropdown(options=['Balanced', 'Minimize carbon', 'Minimize cost', 'Maximize performance', 'Low hardware'], value='Balanced', description='Priority', style=style, layout=layout)
    apply_scenario_button = widgets.Button(description='Apply Scenario', icon='sliders')
    deploy_button = widgets.Button(description='Rank Configurations', button_style='primary', icon='check')
    deploy_out = widgets.Output()

    def on_apply_scenario(_):
        scenario = SCENARIOS[scenario_w.value]
        traffic_w.value = scenario['requests_per_day']
        grid_w.value = scenario['grid']
        deploy_input_w.value = scenario['input_condition']
        max_latency_w.value = scenario['max_latency_ms']
        priority_w.value = scenario['priority']
        if scenario['required_f1_percent'] is not None:
            required_f1_w.value = scenario['required_f1_percent']
        with deploy_out:
            clear_output(wait=True)
            display(Markdown(f"**Scenario loaded:** {scenario_w.value}. {scenario['note']} Constraint rule: {scenario['rule']}."))

    def on_deploy(_):
        with deploy_out:
            clear_output(wait=True)
            df = all_configurations(deploy_input_w.value)
            required = scenario_required_f1_percent(df, scenario_w.value, required_f1_w.value)
            ranked = rank_configurations(
                df,
                traffic_w.value,
                years_w.value,
                grid_w.value,
                required,
                max_latency_w.value,
                priority_w.value,
                custom_co2_w.value,
                custom_price_w.value,
            )
            cols = [
                'Rank', 'Feasible', 'Model', 'Quantization', 'Pooling', 'Batch size', 'Input condition',
                'F1_percent', 'Latency_ms', 'Throughput_requests_per_s', 'Energy_Wh_per_1000_requests',
                'Memory_GB', 'Period energy kWh', 'Period CO2e kg', 'Estimated compute cost USD', 'Advisor score'
            ]
            display(Markdown(f"### Deployment simulator results\nRequired F1: **{required:.2f}%**. Maximum latency: **{max_latency_w.value} ms**. Priority: **{priority_w.value}**."))
            display(ranked[cols].head(12))
            feasible = ranked[ranked['Feasible']]
            if feasible.empty:
                display(Markdown('No configuration meets both constraints. Relax latency or F1, or reconsider the client requirement.'))
            else:
                top = feasible.iloc[0]
                display(Markdown(
                    f"**Recommended by advisor:** {top['Model']} / {top['Quantization']} / {top['Pooling']} / batch {int(top['Batch size'])}. "
                    f"It uses {top['Period energy kWh']:.2f} kWh, emits {top['Period CO2e kg']:.3f} kg CO2e, "
                    f"costs about ${top['Estimated compute cost USD']:,.2f} in compute, and achieves {top['F1_percent']:.2f}% F1."
                ))

    apply_scenario_button.on_click(on_apply_scenario)
    deploy_button.on_click(on_deploy)
    deployment_tab = widgets.VBox([
        widgets.HTML('<h3>Deployment simulator and advisor</h3>'),
        scenario_w,
        widgets.HBox([apply_scenario_button, deploy_button]),
        widgets.HBox([traffic_w, grid_w]),
        widgets.HBox([custom_co2_w, custom_price_w]),
        widgets.HBox([years_w, deploy_input_w]),
        widgets.HBox([required_f1_w, max_latency_w]),
        priority_w,
        deploy_out,
    ])

    pareto_input_w = widgets.Dropdown(options=INPUT_OPTIONS, value='Clean', description='Input condition', style=style, layout=layout)
    pareto_latency_w = widgets.IntText(value=10_000, description='Max latency ms', style=style, layout=layout)
    pareto_button = widgets.Button(description='Run Pareto Analysis', button_style='primary', icon='line-chart')
    pareto_out = widgets.Output()

    def on_pareto(_):
        with pareto_out:
            clear_output(wait=True)
            df = all_configurations(pareto_input_w.value)
            df = df[df['Latency_ms'] <= pareto_latency_w.value].copy()
            df['Pareto efficient'] = pareto_flags(df)
            display(Markdown(f"### Pareto analysis for {pareto_input_w.value} inputs"))
            display(df.sort_values(['Pareto efficient', 'Energy_Wh_per_1000_requests'], ascending=[False, True])[
                ['Pareto efficient', 'Model', 'Quantization', 'Pooling', 'Batch size', 'F1_percent', 'Latency_ms', 'Energy_Wh_per_1000_requests', 'Memory_GB']
            ].head(20))
            fig, ax = plt.subplots(figsize=(8, 5))
            non = df[~df['Pareto efficient']]
            par = df[df['Pareto efficient']]
            ax.scatter(non['Energy_Wh_per_1000_requests'], non['F1_percent'], alpha=0.35, label='Dominated')
            ax.scatter(par['Energy_Wh_per_1000_requests'], par['F1_percent'], s=90, label='Pareto-efficient')
            for _, row in par.iterrows():
                label = f"{row['Model'][0]}-{row['Quantization']}-B{int(row['Batch size'])}"
                ax.annotate(label, (row['Energy_Wh_per_1000_requests'], row['F1_percent']), xytext=(5, 5), textcoords='offset points', fontsize=8)
            ax.set_xlabel('Energy (Wh / 1,000 requests)')
            ax.set_ylabel('F1 (%)')
            ax.set_title('Accuracy-energy Pareto plot')
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()

    pareto_button.on_click(on_pareto)
    pareto_tab = widgets.VBox([
        widgets.HTML('<h3>Pareto analysis</h3>'),
        widgets.HBox([pareto_input_w, pareto_latency_w]),
        pareto_button,
        pareto_out,
    ])

    template_w = widgets.Dropdown(options=list(EXPERIMENT_TEMPLATES.keys()), value='Group A - Model size', description='Template', style=style, layout=widgets.Layout(width='520px'))
    show_template_button = widgets.Button(description='Show Template', icon='list')
    load_template_button = widgets.Button(description='Run and Save Template', button_style='primary', icon='play')
    template_out = widgets.Output()

    def on_show_template(_):
        with template_out:
            clear_output(wait=True)
            show_template(template_w.value)

    def on_load_template(_):
        with template_out:
            clear_output(wait=True)
            rows = load_template_results(template_w.value)
            display(Markdown(f"Loaded and saved **{len(rows)}** results for **{template_w.value}**."))
            display(compact_results_table(rows))
            display(Markdown('Next step: go to the Saved results tab and compare the results.'))

    show_template_button.on_click(on_show_template)
    load_template_button.on_click(on_load_template)
    templates_tab = widgets.VBox([
        widgets.HTML('<h3>Predefined experiment templates</h3>'),
        template_w,
        widgets.HBox([show_template_button, load_template_button]),
        template_out,
    ])

    tabs = widgets.Tab(children=[experiment_tab, results_tab, deployment_tab, pareto_tab, templates_tab])
    for i, title in enumerate(['Experiment', 'Saved Results', 'Deployment', 'Pareto', 'Templates']):
        tabs.set_title(i, title)
    display(tabs)


In [ ]:
#@title Launch AI Deployment Workbench
launch_workbench()


## Semester structure

| Weeks | Phase | What you do |
|---:|---|---|
| 1-2 | AI + sustainability introduction | Learn only the concepts required to interpret model size, quantization, batching, F1, latency, energy, carbon, and cost. |
| 3 | Baseline laboratory | Everyone runs the same supplied baseline and learns how to use the notebook. |
| 4-7 | Controlled experiments | Teams investigate one assigned design dimension using predefined experiment templates. |
| 8-9 | Sustainability analysis | Convert measured results into annual energy, carbon, and cost under different deployment scales. |
| 10-11 | Design scenarios | Choose configurations for fictional customers with conflicting requirements. |
| 12 | Optimization / Pareto analysis | Use automated decision tools to identify efficient solutions. |
| 13 | Final engineering design | Recommend and defend a sustainable AI deployment. |
| 14 | Presentation | Present the solution as a design review, not an ML paper. |


## Team experiments

| Group | Design dimension | Main comparison | Engineering question |
|---|---|---|---|
| A | Model size | Small / Medium / Large | When is additional model complexity worth paying for? |
| B | Quantization | FP16 / INT8 / INT4 | How much computational cost can be removed before quality degrades too much? |
| C | Batching | Batch 1 / 8 / 32 | How does serving many requests together affect energy, latency, and throughput? |
| D | Robustness | Pooling under clean and noisy input | Does the most energy-efficient model remain effective when real inputs become messy? |
| E | Deployment scale | Same configurations at 10k, 1M, 10M, and 100M requests/year | When does a small efficiency improvement become environmentally important? |

All five groups can start simultaneously. Nobody depends on another team's results.


## Client scenarios

Use the **Deployment** tab to load and analyze these customers.

| Client | Decision pressure |
|---|---|
| Hospital document-routing system | Accuracy is extremely important. Volume is moderate. Energy matters, but F1 cannot fall below 98% of the best available configuration for the input condition. |
| High-volume e-commerce classifier | 100 million requests/year. A 1 F1-point difference is acceptable. Energy and cost are important. |
| Small local manufacturer | Very low request volume. Wants local deployment and low hardware requirements. |
| Public-sector service in Quebec | Low-carbon electricity, moderate traffic, strong sustainability requirements. |
| Cloud application on carbon-intensive grid | Large traffic volume and high carbon intensity. |

There may be no universally best model. A configuration that is sustainable for one scenario can be a poor choice for another.


## Final assignment

Act as a sustainable AI consulting team. Your client gives you a final deployment specification. Recommend one AI deployment configuration and defend it.

Your report should answer four questions:

| Question | What to include |
|---|---|
| Technical | Which AI configuration should be deployed? |
| Environmental | What are its annual energy and carbon impacts? |
| Economic | What does it cost compared with alternatives? |
| Design justification | Why is this solution preferable to alternatives given the client's constraints? |

Your final presentation should include an automatically generated Pareto plot, identify dominated and Pareto-efficient solutions, and explain the engineering choice. You do not need to explain transformer architecture.


# Fallback Colab Forms

Use these cells only if the interactive workbench above does not display correctly. In Colab, the code is hidden and you only change form fields.


In [ ]:
#@title RUN EXPERIMENT - fallback form
model = "Medium" #@param ["Small", "Medium", "Large"]
quantization = "INT8" #@param ["FP16", "INT8", "INT4"]
pooling = "Mean" #@param ["Last-token", "Mean", "Attention"]
batch_size = 8 #@param [1, 8, 32] {type:"raw"}
input_condition = "Clean" #@param ["Clean", "Mild noise", "Heavy noise"]

LATEST_RESULT = evaluate_config(model, quantization, pooling, batch_size, input_condition)
display(metric_table(LATEST_RESULT))


In [ ]:
#@title SAVE RESULT - fallback form
save_latest_result = True #@param {type:"boolean"}

latest_result = globals().get('LATEST_RESULT')

if save_latest_result and save_result(latest_result):
    display(Markdown(f"Saved result {len(SAVED_RESULTS)}."))
else:
    display(Markdown('No result saved. Run an experiment first.'))


In [ ]:
#@title COMPARE EXPERIMENTS - fallback form
show_plots = True #@param {type:"boolean"}

df = compact_results_table(SAVED_RESULTS)
if df.empty:
    display(Markdown('No saved results yet.'))
else:
    display(df)
    if show_plots:
        plot_saved_results(df)


In [ ]:
#@title DEPLOYMENT SIMULATOR - fallback form
scenario = "Custom" #@param ["Custom", "Hospital document-routing system", "High-volume e-commerce classifier", "Small local manufacturer", "Public-sector service in Quebec", "Cloud application on carbon-intensive grid"]
traffic_requests_per_day = 1000000 #@param {type:"integer"}
grid = "Ontario" #@param ["Quebec", "Ontario", "Alberta", "Custom"]
custom_kg_co2e_per_kwh = 0.2 #@param {type:"number"}
custom_electricity_usd_per_kwh = 0.1 #@param {type:"number"}
operating_period_years = 1.0 #@param {type:"number"}
input_condition = "Clean" #@param ["Clean", "Mild noise", "Heavy noise"]
required_f1_percent = 93.0 #@param {type:"number"}
maximum_latency_ms = 100 #@param {type:"integer"}
priority = "Balanced" #@param ["Balanced", "Minimize carbon", "Minimize cost", "Maximize performance", "Low hardware"]

if scenario != 'Custom':
    s = SCENARIOS[scenario]
    traffic_requests_per_day = s['requests_per_day']
    grid = s['grid']
    input_condition = s['input_condition']
    maximum_latency_ms = s['max_latency_ms']
    priority = s['priority']

df = all_configurations(input_condition)
required = scenario_required_f1_percent(df, scenario, required_f1_percent)
ranked = rank_configurations(df, traffic_requests_per_day, operating_period_years, grid, required, maximum_latency_ms, priority, custom_kg_co2e_per_kwh, custom_electricity_usd_per_kwh)
cols = ['Rank', 'Feasible', 'Model', 'Quantization', 'Pooling', 'Batch size', 'Input condition', 'F1_percent', 'Latency_ms', 'Energy_Wh_per_1000_requests', 'Memory_GB', 'Period energy kWh', 'Period CO2e kg', 'Estimated compute cost USD', 'Advisor score']
display(Markdown(f"Required F1: **{required:.2f}%**. Maximum latency: **{maximum_latency_ms} ms**. Priority: **{priority}**."))
display(ranked[cols].head(12))


In [ ]:
#@title PARETO ANALYSIS - fallback form
input_condition = "Clean" #@param ["Clean", "Mild noise", "Heavy noise"]
maximum_latency_ms = 10000 #@param {type:"integer"}

df = all_configurations(input_condition)
df = df[df['Latency_ms'] <= maximum_latency_ms].copy()
df['Pareto efficient'] = pareto_flags(df)
display(df.sort_values(['Pareto efficient', 'Energy_Wh_per_1000_requests'], ascending=[False, True])[
    ['Pareto efficient', 'Model', 'Quantization', 'Pooling', 'Batch size', 'F1_percent', 'Latency_ms', 'Energy_Wh_per_1000_requests', 'Memory_GB']
].head(20))

fig, ax = plt.subplots(figsize=(8, 5))
non = df[~df['Pareto efficient']]
par = df[df['Pareto efficient']]
ax.scatter(non['Energy_Wh_per_1000_requests'], non['F1_percent'], alpha=0.35, label='Dominated')
ax.scatter(par['Energy_Wh_per_1000_requests'], par['F1_percent'], s=90, label='Pareto-efficient')
for _, row in par.iterrows():
    ax.annotate(f"{row['Model'][0]}-{row['Quantization']}-B{int(row['Batch size'])}", (row['Energy_Wh_per_1000_requests'], row['F1_percent']), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel('Energy (Wh / 1,000 requests)')
ax.set_ylabel('F1 (%)')
ax.set_title('Accuracy-energy Pareto plot')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
#@title EXPERIMENT TEMPLATES - fallback form
template = "Group A - Model size" #@param ["Group A - Model size", "Group B - Quantization", "Group C - Batching", "Group D - Robustness", "Group E - Deployment scale"]
run_and_save_template = False #@param {type:"boolean"}

show_template(template)
if run_and_save_template:
    rows = load_template_results(template)
    display(Markdown(f"Loaded and saved **{len(rows)}** results for **{template}**."))
    display(compact_results_table(rows))
